In [ ]:
# Step 2: Data Preprocessing for Unsupervised Learning

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

# --- Load Data ---
display(Markdown("## Step 2: Data Preprocessing"))
df = pd.read_csv("../data/retail_customer_data.csv")

# --- Separate columns ---
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()

# --- Identify highly skewed numeric features ---
skewness = df[numeric_cols].skew().sort_values(ascending=False)
skewed_features = skewness[abs(skewness) > 1.0].index.tolist()
normal_features = list(set(numeric_cols) - set(skewed_features))

# --- Imputation ---
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

# --- Transformation pipelines ---
numeric_pipeline = Pipeline([
    ("imputer", numeric_imputer),
    ("scaler", StandardScaler())
])

skewed_pipeline = Pipeline([
    ("imputer", numeric_imputer),
    ("power", PowerTransformer(method='yeo-johnson')),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", categorical_imputer),
    ("encoder", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# --- Combine transformers ---
preprocessor = ColumnTransformer(transformers=[
    ("normal_num", numeric_pipeline, normal_features),
    ("skewed_num", skewed_pipeline, skewed_features),
    ("cat", categorical_pipeline, categorical_cols)
])

# --- Apply transformations ---
processed_array = preprocessor.fit_transform(df)
processed_feature_names = (
    normal_features + skewed_features +
    list(preprocessor.named_transformers_["cat"].named_steps["encoder"].get_feature_names_out(categorical_cols))
)

# --- Create final DataFrame ---
processed_df = pd.DataFrame(processed_array, columns=processed_feature_names)
processed_df.index = df.index

# --- Save cleaned data ---
os.makedirs("../data/processed", exist_ok=True)
processed_df.to_csv("../data/processed/retail_customer_cleaned.csv", index=False)

# --- Report ---
display(Markdown("### ✅ Preprocessing Complete"))
display(Markdown(f"Processed dataset shape: **{processed_df.shape}**"))
display(Markdown("Saved to: `../data/processed/retail_customer_cleaned.csv`"))
print("✅ Step 2: Data preprocessing completed successfully!")
print("📁 Processed data saved to ../data/processed/")